In [ ]:
import geopandas as gpd
import osmnx as ox

import focalpy

In [ ]:
agglom_extent_filepath = "../data/raw/zurich-extent.gpkg"
sites_gdf_filepath = "../data/processed/stations.gpkg"
lake_nominatim_query = "Zürichsee"

buildings_gdf_filepath = "../data/processed/buildings.gpkg"
tree_canopy_filepath = "../data/processed/tree-canopy.tif"
tree_val = 1
dem_filepath = "../data/processed/dem.tif"
buffer_dists = [100, 250, 500]
topo_buffer_dists = [250, 500]

# number of sites processed per batch; reduce if running out of memory
chunk_size = 500

dst_filepath = "../data/processed/stations-features.csv"

In [ ]:
region = gpd.read_file(agglom_extent_filepath).buffer(buffer_dists[-1]).iloc[:1]
lake_extent_gdf = ox.geocode_to_gdf(lake_nominatim_query).to_crs(region.crs)[
    ["geometry"]
]
buildings_gdf = gpd.read_file(buildings_gdf_filepath)
# add a "volume" column
buildings_gdf["volume"] = buildings_gdf.geometry.area * buildings_gdf["height"]

In [ ]:
# `site_gser` should already be in EPSG:2056
site_gser = gpd.read_file(sites_gdf_filepath)
sites_index_name = site_gser.columns.drop("geometry")[0]
site_gser = site_gser.set_index(sites_index_name)["geometry"]

In [ ]:
fa = focalpy.FocalAnalysis(
    [buildings_gdf, tree_canopy_filepath, dem_filepath, lake_extent_gdf, dem_filepath],
    site_gser,
    [buffer_dists, buffer_dists, [0], [0], topo_buffer_dists],
    [
        "compute_vector_features",
        "compute_raster_features",
        "compute_elevation_at_site",
        "compute_vector_distance",
        "compute_terrain_attributes",
    ],
    feature_col_prefixes=["building", "tree", "", "lake", ""],
    feature_methods_args={
        "compute_vector_features": [{"volume": "sum"}],
        "compute_terrain_attributes": [
            ["aspect", "slope", "topographic_position_index"]
        ],
    },
    feature_methods_kwargs={
        "compute_raster_features": {"stats": "sum"},
        "compute_terrain_attributes": {"stats": "mean"},
    },
    chunk_size=chunk_size,
)

  0%|          | 0/118 [00:00<?, ?it/s]

  0%|          | 0/118 [00:00<?, ?it/s]

In [ ]:
fa.features_df.head()

,building_volume_sum_100,building_volume_sum_250,building_volume_sum_500,tree_sum_100,tree_sum_250,tree_sum_500,elevation_0,lake_distance_0,aspect_mean_250,aspect_mean_500,slope_mean_250,slope_mean_500,topographic_position_index_mean_250,topographic_position_index_mean_500
station_id,,,,,,,,,,,,,,
KLO,0.047963,4.796252e-02,4.239974e+03,0.0,0.0,2091.0,426.107239,12548.878257,154.824131,145.856184,1.006813,1.756003,0.000013,0.000094
REH,53579.327844,1.289910e+05,3.275898e+05,690.0,5862.0,29177.0,443.399414,7037.797441,177.117733,174.696470,2.149442,3.195710,0.000452,-0.000174
SMA,62952.852086,3.301923e+05,1.019969e+06,4994.0,31862.0,125308.0,555.867859,2046.655244,211.169714,210.835274,10.137564,10.919506,0.000075,-0.000143
Zürich-Kaserne,263396.449315,2.060322e+06,5.209396e+06,1295.0,5673.0,26116.0,406.533997,1495.165182,171.997938,170.120588,2.356884,3.759559,0.000025,0.000003
Dübendorf-Empa,113076.299845,6.894265e+05,1.611062e+06,1483.0,9898.0,57244.0,430.772980,6439.998871,193.932043,192.267062,5.066118,5.500230,-0.000452,-0.000036


In [ ]:
fa.features_df.to_csv(dst_filepath)